In [20]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

In [21]:
df = pd.read_csv(
    "../data/processed/feature_matrix_geo_v2.csv"
)

print(df.shape)

df.head()

(4964, 13)


,lat,lon,mean_annual_rf,std_annual,cv,dry_days,heavy_days,elevation,soil_clay,soil_soc,slope,water_occurrence,water_norm
0,8.25,77.00,1368.70980,409.68338,0.299321,253.051282,0.487179,0.0,0,NaN,127.199997,100.0,1.00
1,8.25,77.25,1163.10000,317.94876,0.273363,265.871795,0.487179,80.0,346,NaN,120.199997,0.0,0.00
2,8.25,77.50,749.88873,343.41086,0.457949,313.076923,0.282051,71.0,346,NaN,134.000000,12.0,0.12
3,8.25,77.75,758.49920,328.97153,0.433714,294.102564,0.256410,24.0,330,NaN,75.000000,0.0,0.00
4,8.50,76.75,1833.17690,363.57733,0.198332,233.205128,0.820513,0.0,0,NaN,28.400000,100.0,1.00


In [22]:
print(df.columns.tolist())

['lat', 'lon', 'mean_annual_rf', 'std_annual', 'cv', 'dry_days', 'heavy_days', 'elevation', 'soil_clay', 'soil_soc', 'slope', 'water_occurrence', 'water_norm']


In [23]:
df["soil_soc"] = df["soil_soc"].fillna(
    df["soil_soc"].median()
)

df["soil_clay"] = df["soil_clay"].fillna(
    df["soil_clay"].median()
)

df["water_occurrence"] = (
    df["water_occurrence"]
    .replace(255, np.nan)
    .fillna(0)
)

In [24]:
scaler = MinMaxScaler()

df["cv_norm"] = scaler.fit_transform(
    df[["cv"]]
)

df["dry_norm"] = scaler.fit_transform(
    df[["dry_days"]]
)

df["heavy_norm"] = scaler.fit_transform(
    df[["heavy_days"]]
)

df["slope_norm"] = scaler.fit_transform(
    df[["slope"]]
)

df["soc_norm"] = scaler.fit_transform(
    df[["soil_soc"]]
)

df["clay_norm"] = scaler.fit_transform(
    df[["soil_clay"]]
)

df["water_norm"] = (
    df["water_occurrence"] / 100
)

In [25]:
df["rainfall_score"] = (

    1 -

    (

        0.4*df["cv_norm"]

        +

        0.3*df["dry_norm"]

        +

        0.3*df["heavy_norm"]

    )

)

In [26]:
df["soil_score"] = (

    0.6*df["soc_norm"]

    +

    0.4*(1-df["clay_norm"])

)

In [27]:
df["terrain_score"] = (

    1 - df["slope_norm"]

)

In [28]:
df["permeability"] = (

    0.6*(1-df["clay_norm"])

    +

    0.4*df["soc_norm"]

)

df["permeability_norm"] = scaler.fit_transform(
    df[["permeability"]]
)

In [29]:
df["vegetation_proxy"] = (

    0.7*df["soc_norm"]

    +

    0.3*df["water_norm"]

)

In [30]:
df["scri"] = (

    0.25*df["rainfall_score"]

    +

    0.20*df["soil_score"]

    +

    0.15*df["terrain_score"]

    +

    0.15*df["water_norm"]

    +

    0.15*df["permeability_norm"]

    +

    0.10*df["vegetation_proxy"]

)

df["scri"] *= 100

In [31]:
df["scri_class"] = pd.cut(

    df["scri"],

    bins=[0,40,50,60,100],

    labels=[
        "Poor",
        "Moderate",
        "Good",
        "Excellent"
    ]

)

In [32]:
print(df["scri"].describe())

print()

print(df["scri_class"].value_counts())

count    4964.000000
mean       47.686875
std         6.457108
min        32.817361
25%        43.808951
50%        46.219498
75%        49.711761
max        78.782501
Name: scri, dtype: float64

scri_class
Moderate     3545
Good          904
Excellent     282
Poor          233
Name: count, dtype: int64


In [33]:
df.to_csv(

    "../dashboard/scri_dashboard_data.csv",

    index=False

)

df.to_csv(

    "../data/processed/scri_v2_final.csv",

    index=False

)

In [34]:
print(df["scri_class"].value_counts())
print(df.columns.tolist())

scri_class
Moderate     3545
Good          904
Excellent     282
Poor          233
Name: count, dtype: int64
['lat', 'lon', 'mean_annual_rf', 'std_annual', 'cv', 'dry_days', 'heavy_days', 'elevation', 'soil_clay', 'soil_soc', 'slope', 'water_occurrence', 'water_norm', 'cv_norm', 'dry_norm', 'heavy_norm', 'slope_norm', 'soc_norm', 'clay_norm', 'rainfall_score', 'soil_score', 'terrain_score', 'permeability', 'permeability_norm', 'vegetation_proxy', 'scri', 'scri_class']


In [35]:
df.to_csv(
    "../dashboard/scri_dashboard_data.csv",
    index=False
)

print("saved")

saved


In [36]:
priority = df.nsmallest(
    100,
    "scri"
)

priority.to_csv(
    "../dashboard/priority_interventions.csv",
    index=False
)

priority.head()

,lat,lon,mean_annual_rf,std_annual,cv,dry_days,heavy_days,elevation,soil_clay,soil_soc,...,soc_norm,clay_norm,rainfall_score,soil_score,terrain_score,permeability,permeability_norm,vegetation_proxy,scri,scri_class
2436,23.75,91.00,37.212044,229.39044,6.164414,362.589744,0.025641,8.0,328,174.0,...,0.141348,0.620038,0.300450,0.236794,0.989854,0.284517,0.315532,0.098944,32.817361,Poor
3237,26.00,89.50,54.514240,336.04837,6.164414,362.205128,0.051282,33.0,300,214.0,...,0.173842,0.567108,0.300748,0.277462,0.999426,0.329272,0.372708,0.121690,34.866850,Poor
2621,24.25,91.25,48.156820,296.85858,6.164414,362.051282,0.025641,10.0,286,203.0,...,0.164907,0.540643,0.301250,0.282687,0.985929,0.341577,0.388428,0.115435,34.954679,Poor
2622,24.25,91.50,50.707275,312.58063,6.164414,362.205128,0.051282,16.0,272,210.0,...,0.170593,0.514178,0.300748,0.296685,0.958457,0.359731,0.411619,0.119415,35.197685,Poor
4924,36.50,72.75,376.257900,277.84857,0.738452,349.025641,0.589744,2526.0,314,259.0,...,0.210398,0.593573,0.674106,0.288810,0.425290,0.328016,0.371103,0.147279,36.047523,Poor


In [37]:
top10 = df.nsmallest(
    10,
    "scri"
)

top10[
    [
        "lat",
        "lon",
        "scri",
        "scri_class"
    ]
]

,lat,lon,scri,scri_class
2436,23.75,91.00,32.817361,Poor
3237,26.00,89.50,34.866850,Poor
2621,24.25,91.25,34.954679,Poor
2622,24.25,91.50,35.197685,Poor
4924,36.50,72.75,36.047523,Poor
1101,19.25,75.25,36.048868,Poor
1368,20.50,76.00,36.168142,Poor
1316,20.25,77.00,36.197146,Poor
1151,19.50,75.50,36.204395,Poor
140,11.25,76.50,36.357298,Poor


In [38]:
top10.to_csv(
    "../dashboard/top10_hotspots.csv",
    index=False
)